# DeBERTa Fine-Tuning for Team Morale Prediction

This notebook fine-tunes `microsoft/deberta-v3-base` for regression on a custom football morale dataset. The model predicts a morale score from 1 to 10 based on recent football news headlines and team context.

The workflow is designed for Google Colab with GPU acceleration. The final model is exported as a zipped `deberta-morale-final` folder for use in the local Streamlit application.

In [40]:
!pip install transformers==4.44.0
!pip install datasets torch scikit-learn

## 2. Imports

Load the data science, PyTorch, Hugging Face Datasets, Transformers, and Hugging Face Hub utilities used throughout the notebook.

In [41]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, Value
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from huggingface_hub import login

## 3. Hugging Face Authentication

Authenticate with Hugging Face Hub if required by the runtime. The base DeBERTa model is public, but authentication can help avoid access or rate-limit issues in Colab.

In [42]:
import os
from dotenv import load_dotenv
load_dotenv()
login(token=os.environ.get("HF_TOKEN"))

## 4. Data Loading

Mount Google Drive and load the prepared train and validation CSV files. Each file contains `text` inputs built from football news headlines and a numeric `label` representing morale on a 1-10 scale.

In [43]:
from google.colab import drive
drive.mount('/content/drive')

train_df = pd.read_csv('/content/drive/MyDrive/sports_prediction/train.csv')
val_df = pd.read_csv('/content/drive/MyDrive/sports_prediction/val.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Initial Data Checks

Confirm that the loaded splits have valid label columns and no missing label values before any preprocessing is applied.

In [44]:
print(f"Train NaN labels: {train_df['label'].isna().sum()} / {len(train_df)}")
print(f"Val NaN labels:   {val_df['label'].isna().sum()} / {len(val_df)}")

Train NaN labels: 0 / 8800
Val NaN labels:   0 / 2200


### Data Cleaning

Drop rows with missing `text` or `label` values to avoid tokenizer failures and invalid regression targets during training.

In [45]:
train_df = train_df.dropna(subset=['label', 'text'])
val_df = val_df.dropna(subset=['label', 'text'])

### Label Normalization

Scale labels from `[1, 10]` to `[0.1, 1.0]` by dividing by `LABEL_MAX = 10.0`. Keeping labels in a compact numeric range improves regression stability during fine-tuning.

In [46]:
LABEL_MAX = 10.0

train_df['label'] = train_df['label'].astype(np.float32)/LABEL_MAX
val_df['label'] = val_df['label'].astype(np.float32)/LABEL_MAX

### Normalized Label Check

Verify the number of examples after cleaning and confirm that normalized labels remain within the expected range.

In [47]:
print(f"\nPo czyszczeniu — Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Label range train: {train_df['label'].min():.2f} – {train_df['label'].max():.2f}")
print(f"Label range val:   {val_df['label'].min():.2f} – {val_df['label'].max():.2f}")


Po czyszczeniu — Train: 8800, Val: 2200
Label range train: 0.10 – 1.00
Label range val:   0.10 – 1.00


## 5. Tokenizer & Tokenization

Load the `microsoft/deberta-v3-base` tokenizer and define a batch tokenization function. Inputs are padded or truncated to 256 tokens, which is sufficient for compact headline-based morale examples.

In [48]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch['text'], truncation = True, padding = "max_length", max_length = 256)

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


### Dataset Preparation

Convert the pandas DataFrames into Hugging Face `Dataset` objects, tokenize the text in batches, and rename `label` to `labels` so the Trainer can use it as the regression target.

In [49]:
train_dataset = Dataset.from_pandas(train_df).map(tokenize, batched = True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize, batched = True)

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")

Map:   0%|          | 0/8800 [00:00<?, ? examples/s]

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [50]:
train_dataset = train_dataset.cast_column("labels", Value("float32"))
val_dataset = val_dataset.cast_column("labels", Value("float32"))

Casting the dataset:   0%|          | 0/8800 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [51]:
train_dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "labels"])

### Data Validation

Run final sanity checks on the prepared label tensors. The training run should only start if labels contain no `NaN` or infinite values and are stored as `float32` tensors.

In [52]:
train_labels = torch.tensor(train_dataset['labels'])
val_labels = torch.tensor(val_dataset['labels'])
print(f"\nTensor NaN check — train: {torch.isnan(train_labels).sum()}, val: {torch.isnan(val_labels).sum()}")
print(f"Tensor Inf check — train: {torch.isinf(train_labels).sum()}, val: {torch.isinf(val_labels).sum()}")


Tensor NaN check — train: 0, val: 0
Tensor Inf check — train: 0, val: 0


In [53]:
print(f"Label dtype train: {train_dataset[0]['labels'].dtype}")
print(f"Label dtype val: {val_dataset[0]['labels'].dtype}")

Label dtype train: torch.float32
Label dtype val: torch.float32


## 6. Model

Load `microsoft/deberta-v3-base` with a single-output sequence classification head. With `num_labels=1`, the head is trained as a regression layer for morale prediction.

In [54]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 1)
print(f"\nGPU is available: {torch.cuda.is_available()}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



GPU is available: True


## 7. Evaluation Metrics

Define MSE and MAE on the original 1-10 morale scale. MAE is the main selection metric because it is directly interpretable as the average prediction error in morale points.

In [56]:
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  predictions = predictions.squeeze()

  pred_orig = predictions * LABEL_MAX
  labels_orig = labels * LABEL_MAX
  mse = np.mean((pred_orig - labels_orig) ** 2)
  mae = np.mean(np.abs(pred_orig - labels_orig))
  return {"mse": float(mse), "mae": float(mae)}

## 8. Training Configuration

Configure the Hugging Face `Trainer`. The setup evaluates and saves once per epoch, tracks the best checkpoint by validation MAE, and uses early stopping to avoid unnecessary training once validation quality stops improving.

In [57]:
from transformers import EarlyStoppingCallback
args = TrainingArguments(
    output_dir = "./deberta-morale",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    metric_for_best_model = "mae",
    logging_steps=10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    warmup_ratio=0.1,
    weight_decay = 0.01,
    learning_rate = 2e-5,
    max_grad_norm = 1.0,
    load_best_model_at_end = True,
    fp16=False,
    bf16=False,
    greater_is_better= False,
)
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],

)

## 9. Training

Start fine-tuning. During training, validation metrics are reported after each epoch; the best checkpoint is kept according to the lowest validation MAE.

In [58]:
trainer.train()

Epoch,Training Loss,Validation Loss,Mse,Mae
1,0.017900,0.016613,1.661310,1.084231
2,0.012900,0.015327,1.532690,1.047970
3,0.011700,0.013548,1.354798,1.011066
4,0.012700,0.013941,1.394100,1.019208
5,0.010200,0.015091,1.509093,1.045793


TrainOutput(global_step=2750, training_loss=0.019921618675643748, metrics={'train_runtime': 3099.3046, 'train_samples_per_second': 14.197, 'train_steps_per_second': 0.887, 'total_flos': 5788495054848000.0, 'train_loss': 0.019921618675643748, 'epoch': 5.0})

## 10. Save & Export

Save the best fine-tuned model and tokenizer to `./deberta-morale-final`, archive the folder, and download the zip file for replacement in the local project under `models/deberta-morale-final/`.

In [59]:
trainer.save_model("./deberta-morale-final")
tokenizer.save_pretrained('./deberta-morale-final')

import shutil
from google.colab import files

shutil.make_archive("deberta-morale-final", 'zip', "./deberta-morale-final")
files.download("deberta-morale-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>